In [1]:
import os
import re
import string
import pickle
import zipfile
import logging
import urllib.request
import random
import json
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import nltk
for resource, pkg in [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("corpora/stopwords", "stopwords"),
]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords as nltk_stopwords
from nltk.tokenize import word_tokenize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

# ── Logging ───────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler("../training.log", mode="w"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Train")

Device : cpu


In [2]:
CONFIG = {
    "data_dir"            : Path("../data"),
    "glove_dir"           : Path("../data/glove"),
    "dataset_url"         : "https://huggingface.co/datasets/SetFit/bbc-news/raw/main/bbc-text.csv",
    "dataset_local"       : Path("../data/bbc-text.csv"),
    "glove_url"           : "https://nlp.stanford.edu/data/glove.6B.zip",
    "glove_zip"           : Path("../data/glove/glove.6B.zip"),
    "glove_file"          : Path("../data/glove/glove.6B.100d.txt"),
    "model_save_path"     : Path("../saved_model.pth"),
    "word2idx_save_path"  : Path("../models/word2idx.pkl"),
    "label_enc_save_path" : Path("../models/label_encoder.pkl"),
    "train_ratio"         : 0.80,
    "val_ratio"           : 0.10,
    "test_ratio"          : 0.10,
    "random_seed"         : SEED,
    "max_vocab_size"      : 25_000,
    "max_seq_len"         : 512,
    "min_word_freq"       : 2,
    "embed_dim"           : 100,
    "fine_tune_embeddings": True,
    "hidden_size"         : 128,
    "num_layers"          : 2,
    "bidirectional"       : True,
    "lstm_dropout"        : 0.5,
    "fc_dropout"          : 0.3,
    "num_classes"         : 5,
    "epochs"              : 20,
    "batch_size"          : 64,
    "learning_rate"       : 1e-3,
    "weight_decay"        : 1e-4,
    "grad_clip"           : 1.0,
    "early_stop_patience" : 5,
    "scheduler_T_max"     : 20,
    "label_map"           : {
        "business": "Business", "entertainment": "Entertainment",
        "politics": "Politics", "sport": "Sports", "tech": "Technology",
    },
}
for key in ["data_dir", "glove_dir"]:
    CONFIG[key].mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)
print(" Config ready.")

 Config ready.


In [3]:
# ── Load dataset ─────────────────────────────────────────────
if CONFIG["dataset_local"].exists():
    df = pd.read_csv(CONFIG["dataset_local"])
else:
    df = pd.read_csv(CONFIG["dataset_url"])
    df.to_csv(CONFIG["dataset_local"], index=False)
df.columns = [c.strip().lower() for c in df.columns]
df["label"] = df["category"].map(CONFIG["label_map"])
df.dropna(subset=["text", "label"], inplace=True)
df.reset_index(drop=True, inplace=True)
logger.info(f"Loaded {len(df)} rows.")

# ── Preprocessing ─────────────────────────────────────────────
STOP_WORDS = set(nltk_stopwords.words("english"))
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX,  UNK_IDX   = 0, 1

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize_text(text: str) -> List[str]:
    tokens = word_tokenize(text)
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

def preprocess(text: str) -> List[str]:
    return tokenize_text(clean_text(text))

def text_to_indices(tokens: List[str], word2idx: Dict, max_len: int) -> List[int]:
    indices = [word2idx.get(t, UNK_IDX) for t in tokens[:max_len]]
    return indices + [PAD_IDX] * (max_len - len(indices))

print(" Preprocessing ...")
df["tokens"] = df["text"].apply(preprocess)

# ── Label encoding ────────────────────────────────────────────
unique_classes = sorted(df["label"].unique())
STR2IDX: Dict[str, int] = {cls: i for i, cls in enumerate(unique_classes)}
IDX2STR: Dict[int, str] = {i: cls for cls, i in STR2IDX.items()}
df["encoded_label"] = df["label"].map(STR2IDX)

# ── Split ─────────────────────────────────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=(1 - CONFIG["train_ratio"]),
    stratify=df["encoded_label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=temp_df["encoded_label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
logger.info(f"Split — Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}")

# ── Vocabulary ────────────────────────────────────────────────
counter: Counter = Counter()
for tokens in train_df["tokens"]:
    counter.update(tokens)
vocab_words = [w for w, f in counter.most_common(CONFIG["max_vocab_size"]) if f >= CONFIG["min_word_freq"]]
WORD2IDX: Dict[str, int] = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
for w in vocab_words:
    WORD2IDX[w] = len(WORD2IDX)
VOCAB_SIZE = len(WORD2IDX)
logger.info(f"Vocab size: {VOCAB_SIZE:,}")
print(f" Preprocessing done. Vocab: {VOCAB_SIZE:,}")

10:37:27 | INFO | Loaded 2225 rows.


 Preprocessing ...


10:37:30 | INFO | Split — Train:1780 Val:222 Test:223
10:37:31 | INFO | Vocab size: 17,196


 Preprocessing done. Vocab: 17,196


In [4]:
glove_file = CONFIG["glove_file"]
glove_zip  = CONFIG["glove_zip"]

if not glove_file.exists():
    if not glove_zip.exists():
        print("Downloading GloVe (~822 MB) ...")
        def _progress(b, bs, total):
            print(f"\r  {min(b*bs/total*100,100):.1f}%", end="", flush=True)
        urllib.request.urlretrieve(CONFIG["glove_url"], glove_zip, reporthook=_progress)
        print()
    print("Extracting ...")
    with zipfile.ZipFile(glove_zip, "r") as zf:
        zf.extractall(CONFIG["glove_dir"])

print("Loading GloVe vectors ...")
glove_vectors: Dict[str, np.ndarray] = {}
with open(glove_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        parts = line.rstrip().split(" ")
        try:
            vec = np.array(parts[1:], dtype=np.float32)
            if vec.shape[0] == CONFIG["embed_dim"]:
                glove_vectors[parts[0]] = vec
        except ValueError:
            continue
        if (i + 1) % 100_000 == 0:
            print(f"\r  Loaded {i+1:,} lines", end="", flush=True)
print(f"\nLoaded {len(glove_vectors):,} vectors.")

# Build embedding matrix
rng = np.random.default_rng(SEED)
embed_dim = CONFIG["embed_dim"]
EMBEDDING_MATRIX = np.zeros((VOCAB_SIZE, embed_dim), dtype=np.float32)
found = 0
for word, idx in WORD2IDX.items():
    if idx == PAD_IDX:
        continue
    if word in glove_vectors:
        EMBEDDING_MATRIX[idx] = glove_vectors[word]
        found += 1
    else:
        EMBEDDING_MATRIX[idx] = rng.uniform(-0.25, 0.25, embed_dim).astype(np.float32)

logger.info(f"Embedding matrix: {EMBEDDING_MATRIX.shape} | GloVe hits: {found}/{VOCAB_SIZE}")
print(f" Matrix shape: {EMBEDDING_MATRIX.shape} | Coverage: {found/VOCAB_SIZE*100:.1f}%")

Loading GloVe vectors ...
  Loaded 400,000 lines

10:37:45 | INFO | Embedding matrix: (17196, 100) | GloVe hits: 16382/17196



Loaded 400,000 vectors.
 Matrix shape: (17196, 100) | Coverage: 95.3%


In [5]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.labels = labels
        self.sequences = [
            text_to_indices(preprocess(t), word2idx, max_len) for t in texts
        ]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

ML = CONFIG["max_seq_len"]
BS = CONFIG["batch_size"]
train_loader = DataLoader(NewsDataset(train_df["text"].tolist(), train_df["encoded_label"].tolist(), WORD2IDX, ML), batch_size=BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(NewsDataset(val_df["text"].tolist(),   val_df["encoded_label"].tolist(),   WORD2IDX, ML), batch_size=BS, shuffle=False, num_workers=0)
test_loader  = DataLoader(NewsDataset(test_df["text"].tolist(),  test_df["encoded_label"].tolist(),  WORD2IDX, ML), batch_size=BS, shuffle=False, num_workers=0)


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers,
                 num_classes, lstm_dropout, fc_dropout, bidirectional,
                 embedding_matrix=None, fine_tune=True, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if embedding_matrix is not None:
            self.embedding.weight = nn.Parameter(
                torch.tensor(embedding_matrix, dtype=torch.float32), requires_grad=fine_tune
            )
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers,
                            bidirectional=bidirectional, batch_first=True,
                            dropout=lstm_dropout if num_layers > 1 else 0.0)
        dirs = 2 if bidirectional else 1
        self.dropout1 = nn.Dropout(lstm_dropout)
        self.fc1      = nn.Linear(hidden_size * dirs, hidden_size)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(fc_dropout)
        self.fc2      = nn.Linear(hidden_size, num_classes)
        self.bidirectional = bidirectional

    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.lstm(emb)
        h = torch.cat((h[-2], h[-1]), dim=1) if self.bidirectional else h[-1]
        return self.fc2(self.dropout2(self.relu(self.fc1(self.dropout1(h)))))


model = BiLSTMClassifier(
    VOCAB_SIZE, CONFIG["embed_dim"], CONFIG["hidden_size"], CONFIG["num_layers"],
    CONFIG["num_classes"], CONFIG["lstm_dropout"], CONFIG["fc_dropout"],
    CONFIG["bidirectional"], EMBEDDING_MATRIX, CONFIG["fine_tune_embeddings"]
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Model built | Trainable params: {n_params:,}")

 Model built | Trainable params: 2,383,925


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["scheduler_T_max"])

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, patience_count = float("inf"), 0

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for seqs, labels in loader:
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            logits = model(seqs)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                optimizer.step()
            total_loss += loss.item() * seqs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += seqs.size(0)
    return total_loss / total, correct / total

print(f" Training on {DEVICE} for up to {CONFIG['epochs']} epochs ...\n")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-" * 55)

for epoch in range(1, CONFIG["epochs"] + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    print(f"{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | {vl_loss:>8.4f} | {vl_acc:>7.2%}")
    logger.info(f"Epoch {epoch:02d} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f}")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_count = 0
        torch.save(model.state_dict(), CONFIG["model_save_path"])
        print(f"        Checkpoint saved.")
    else:
        patience_count += 1
        if patience_count >= CONFIG["early_stop_patience"]:
            print(f"\n Early stopping at epoch {epoch}.")
            break

print(f"\n Done. Best val loss: {best_val_loss:.4f}")

 Training on cpu for up to 20 epochs ...

Epoch | Train Loss | Train Acc | Val Loss | Val Acc
-------------------------------------------------------


In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")
epochs_ran = list(range(1, len(history["train_loss"]) + 1))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History", fontsize=14, fontweight="bold")

axes[0].plot(epochs_ran, history["train_loss"], "o-", label="Train", linewidth=2)
axes[0].plot(epochs_ran, history["val_loss"],   "s-", label="Val",   linewidth=2)
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(epochs_ran, [a*100 for a in history["train_acc"]], "o-", label="Train", linewidth=2)
axes[1].plot(epochs_ran, [a*100 for a in history["val_acc"]],   "s-", label="Val",   linewidth=2)
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig("../models/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
with open(CONFIG["word2idx_save_path"], "wb") as f:
    pickle.dump(WORD2IDX, f)

with open(CONFIG["label_enc_save_path"], "wb") as f:
    pickle.dump({"str2idx": STR2IDX, "idx2str": IDX2STR}, f)

config_snap = {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}
with open("../models/config_snapshot.json", "w") as f:
    json.dump(config_snap, f, indent=2)

print(" All artifacts saved:")
print(f"   {CONFIG['model_save_path']}")
print(f"   {CONFIG['word2idx_save_path']}")
print(f"   {CONFIG['label_enc_save_path']}")
print("   ../models/config_snapshot.json")